In [ ]:
# [auto] project-root setup
import os, sys
from pathlib import Path

# Walk upward to find the project root (folder containing .gitignore)
_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / '.gitignore').exists():
    _p = _p.parent
PROJECT_ROOT = _p

# Switch cwd to the project root so every relative path (Stage1_Exploration/, Refined_Results_v4/, ...) keeps working
os.chdir(PROJECT_ROOT)
# Let the notebooks do `from viz_config import VizConfig`
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
print(f'[setup] PROJECT_ROOT = {PROJECT_ROOT}')


In [ ]:
import pandas as pd
import numpy as np
import os
import joblib
from pysr import PySRRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error

# ==========================================
# 0. Experiment parameters and path configuration
# ==========================================
# Noise levels to sweep (0%..200%) for a full robustness picture
NOISE_LEVELS = [0,0.1,0.2,0.3,0.35,0.40,0.45,0.50,0.55,0.6,0.65,0.7,0.75,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0]

# Output directory (Refined_Results_v5)
BASE_DIR = "Refined_Results_v5" 
if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

FEATURE_NAMES = ['V_in', 'C_in', 'Area', 'Distance']
SCALE = 1e7 # Concentration scaling factor

# Collects R^2 at every noise level - summarised to CSV at the end
summary_list = []

# ==========================================
# 1. Data loading and preprocessing
# ==========================================
print("Loading data and performing a case-wise split...")
df = pd.read_csv('data/train_dataset_ready.csv')

# Concentration pre-scaling
df['C_in'] = df['C_in'] * SCALE
df['C_out'] = df['C_out'] * SCALE

# Split train/test sets by case
# Ensures the test-set cases are completely unseen
unique_cases = df['Case'].unique()
train_cases, test_cases = train_test_split(unique_cases, test_size=0.3, random_state=42)

# Filter the data
train_df = df[df['Case'].isin(train_cases)].copy()
test_df = df[df['Case'].isin(test_cases)].copy()

print(f"Split done. Train cases: {len(train_cases)}, test cases: {len(test_cases)}")
print(f"Train rows: {len(train_df)}, test rows: {len(test_df)}")

# Prepare training data (clean baseline)
# Note: X_train_clean / y_train_clean are the clean baseline
# Noise will be added to y_train_clean inside the loop below
X_train_clean = train_df[FEATURE_NAMES].values
y_train_clean = train_df['C_out'].values

# Prepare test data (ground truth)
# The test set always stays clean so it reflects real-world accuracy
X_test = test_df[FEATURE_NAMES].values
y_test_clean = test_df['C_out'].values

# Keep auxiliary columns (Case ID, Distance) for later per-case plotting
test_cases_col = test_df['Case'].values
test_dist_col = test_df['Distance'].values

# ==========================================
# 2. Automated experiment loop
# ==========================================
# Loop over every noise level and compare Direct PySR, MLP and Hybrid PySR
for noise_pct in NOISE_LEVELS:
    noise_label = f"{int(noise_pct*100)}pct"
    path = os.path.join(BASE_DIR, f"Noise_{noise_label}")
    if not os.path.exists(path): os.makedirs(path)
    
    print(f"\n>>> Running experiment at noise level {noise_pct*100}%")

    # --- A. Build the noisy training set ---
    # Add Gaussian white noise to the target y only - mimics sensor / environmental noise
    np.random.seed(42)
    noise_train = np.random.normal(0, noise_pct * y_train_clean) if noise_pct > 0 else 0
    y_train_noisy = y_train_clean + noise_train
    
    # Build a noisy test set (only for MLP observation - not a final metric)
    noise_test = np.random.normal(0, noise_pct * y_test_clean) if noise_pct > 0 else 0
    y_test_noisy = y_test_clean + noise_test
    
    # Physical constraint: concentration cannot be negative - floor at 1e-6
    y_train_noisy = np.maximum(y_train_noisy, 1e-6)
    y_test_noisy = np.maximum(y_test_noisy, 1e-6)

    # --- B. Task 1: Direct symbolic regression (baseline) ---
    # Train PySR directly on the noisy data - baseline / control
    print(f"    [Task 1] Running direct PySR...")
    # Sub-sample to 5000 points for speed
    pysr_sub_idx = np.random.choice(len(y_train_noisy), 5000, replace=False)
    
    model_direct = PySRRegressor(
        niterations=150,
        populations=30,
        maxsize=25,
        binary_operators=["+", "*", "-", "/"],
        unary_operators=["exp", "log", "sqrt", "square", "inv(x)=1/x"],
        extra_sympy_mappings={'inv': lambda x: 1/x},
        temp_equation_file=True,
        delete_tempfiles=False, # Keep temp files for reference
        verbosity=0
    )
    # training
    model_direct.fit(X_train_clean[pysr_sub_idx], y_train_noisy[pysr_sub_idx], variable_names=FEATURE_NAMES)
    # Save formulas to CSV
    model_direct.equations_.to_csv(os.path.join(path, f"all_formulas_direct_{noise_label}.csv"))
    # Predict on the clean test set
    y_pred_direct = model_direct.predict(X_test)

    # --- C. Task 2: MLP training ---
    # Train an MLP to act as a nonlinear denoiser
    print(f"    [Task 2] Training the MLP...")
    # Input standardisation is essential for the MLP
    scaler_X = StandardScaler()
    X_train_scaled = scaler_X.fit_transform(X_train_clean)
    X_test_scaled = scaler_X.transform(X_test)

    mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64, 32), # 3-layer hidden network
        learning_rate_init=0.001,
        activation='relu',
        max_iter=1000,      
        random_state=42,
        early_stopping=False
    )
    mlp.fit(X_train_scaled, y_train_noisy)

    # Save the loss curve for convergence inspection
    pd.DataFrame(mlp.loss_curve_, columns=['Loss']).to_csv(os.path.join(path, "mlp_loss_history.csv"))
    # prediction
    y_pred_mlp = mlp.predict(X_test_scaled)
    
    # --- D. Task 3: Hybrid framework ---
    # Strategy: first denoise the training data with the MLP,"denoising" then fit the denoised data with PySR
    print(f"    [Task 3] Running hybrid PySR...")
    
    # 1. MLP Denoise: use the trained MLP to predict the training subset
    # Assumption: the MLP has learned the physical manifold, so its output is closer to ground truth than noisy labels
    y_denoised_train_sub = mlp.predict(X_train_scaled[pysr_sub_idx])
    
    # 2. PySR Symbolic regression: fit the denoised labels
    model_hybrid = PySRRegressor(
        niterations=150,
        populations=30,
        maxsize=25,
        binary_operators=["+", "*", "-", "/"],
        unary_operators=["exp", "log", "sqrt", "square", "inv(x)=1/x"],
        extra_sympy_mappings={'inv': lambda x: 1/x},
        temp_equation_file=True,
        delete_tempfiles=False,
        verbosity=0
    )
    model_hybrid.fit(X_train_clean[pysr_sub_idx], y_denoised_train_sub, variable_names=FEATURE_NAMES)
    # Save formulas
    model_hybrid.equations_.to_csv(os.path.join(path, f"all_formulas_hybrid_{noise_label}.csv"))
    # prediction
    y_pred_hybrid = model_hybrid.predict(X_test)

    # --- E. Save predictions ---
    # Save predictions from every method for later parity / sequence plots
    print(f"    [Saving] Saving prediction comparison data...")
    comparison_df = pd.DataFrame({
        'Case': test_cases_col,
        'Distance': test_dist_col,
        'True_Clean': y_test_clean,        # Ground Truth
        'True_Noisy': y_test_noisy,        # Noisy Measurement
        'Pred_Direct_PySR': y_pred_direct, # Baseline Prediction
        'Pred_MLP': y_pred_mlp,            # Intermediate Denoising
        'Pred_Hybrid_PySR': y_pred_hybrid  # Final Hybrid Prediction
    })
    comparison_df.to_csv(os.path.join(path, "predictions_comparison.csv"), index=False)

    # --- F. Metric calculation ---
    r2_direct = r2_score(y_test_clean, y_pred_direct)
    r2_mlp = r2_score(y_test_clean, y_pred_mlp)
    r2_hybrid = r2_score(y_test_clean, y_pred_hybrid)
    
    summary_list.append({
        'Noise_Ratio': noise_pct,
        'R2_Direct_PySR': r2_direct,
        'R2_MLP': r2_mlp,
        'R2_Hybrid_PySR': r2_hybrid
    })

    print(f"    Result: Direct R^2 = {r2_direct:.4f}, Hybrid R^2 = {r2_hybrid:.4f}")
    
    # Persist the trained model objects
    joblib.dump(mlp, os.path.join(path, "mlp_model.pkl"))
    joblib.dump(scaler_X, os.path.join(path, "scaler_X.pkl"))

# ==========================================
# 3. Final summary
# ==========================================
# Save the R^2 summary across noise levels (used to plot Figure 6)
summary_df = pd.DataFrame(summary_list)
summary_df.to_csv(os.path.join(BASE_DIR, "final_summary_r2.csv"), index=False)

print("\n" + "="*50)
print("All experiments complete!")
print(f"1. Line-plot data: {os.path.join(BASE_DIR, 'final_summary_r2.csv')}")
print(f"2. Scatter-plot data: see predictions_comparison.csv inside each Noise_xxx folder")
print("="*50)